# Tool 5bis: Participant selection on sleep macrostructure criteria

Looks at the **distribution of every macrostructure metric** computed by tool 5 (TST, sleep efficiency,
AHI, PLM index, T88...) across the database and lets you **exclude participants on clinical criteria**
(e.g. *AHI > 15/h*, *TST < 240 min*). The thresholds are drawn **live** on the distributions while you
type them, so you see at once who falls outside and how many participants each criterion removes.
When the selection looks right, **validate** it.

**Input**: `reports_features_macrostructure/global_sleep_metrics.tsv` written by tool 5 (one row per
participant). Nothing else is read: no EDF, no hypnogram.

**Rule**: a participant is **excluded** as soon as **one** enabled criterion is violated
(`value < min` or `value > max`: a value equal to the threshold is kept). A missing value (metric not
computed, e.g. no SpO2 channel) does not exclude, unless *Missing value excludes* is ticked.

**Outputs** (written beside the loaded table, in `reports_features_macrostructure/`):
`participant_selection.tsv` (one row per participant: `file_id`, `excluded`, the reason, one
`fail_<metric>` column per criterion), `participant_selection_criteria.tsv` (the criteria, reloaded
next time to pre-fill the thresholds) and `participant_selection_report.html`. This is a **decision
record**: the other tools do not read it (yet).

Work through the sections in order: **1** load the table, **2** set the criteria and watch the
distributions, **3** validate.

In [ ]:
# =============================================================================
# Tool 5bis - Participant selection: setup, constants and shared configuration
# =============================================================================
import os
import io
import html
import base64
import datetime
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from ipyfilechooser import FileChooser

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.max_open_warning'] = 0

# Show the winking unicorn when the selection is validated (set to False for a serious audience)
SHOW_UNICORN = True
UNICORN_GIF_NAME = 'unicorn_wink.gif'      # generated by tools/images/_make_unicorn_gif.py

# Input / output names
GLOBAL_TABLE_NAME = 'global_sleep_metrics.tsv'
REPORTS_DIRNAME = 'reports_features_macrostructure'
SELECTION_TSV_NAME = 'participant_selection.tsv'
CRITERIA_TSV_NAME = 'participant_selection_criteria.tsv'
REPORT_HTML_NAME = 'participant_selection_report.html'

# Group order of the criteria accordion ('other' = numeric columns tool 5 does not register,
# e.g. columns of an older tool-5 run or participant-table columns such as age)
GROUP_ORDER = ['macrostructure', 'respiratory', 'plm', 'arousal', 'odi', 'spo2', 'other']

# Figure colours: kept participants in secondary ink, excluded ones in the "critical" status red
# (always shown with a legend, never colour alone)
COLOR_KEPT = '#52514e'
COLOR_EXCLUDED = '#d03b3b'
COLOR_HIST = '#d9d8d2'
COLOR_BAND = '#2e9e5b'          # indicative range (shaded lightly)
COLOR_MUTED = '#898781'

# -----------------------------------------------------------------------------
# COPIED FROM tools/5_sleep_macrostructure_voila.ipynb (setup cell): keep in sync
# -----------------------------------------------------------------------------
# --- metric registry ----------------------------------------------------------
# (key, label, unit, group, one-line definition) - the single source for the long TSV, the
# glossary, the workbook and the report. Order = column order of the global table.
METRICS = [
    ('tib_min', 'Time in bed (TIB)', 'min', 'macrostructure',
     'Lights-off to lights-on: number of epochs in bed x epoch length.'),
    ('sol_min', 'Sleep onset latency (SOL)', 'min', 'macrostructure',
     'Lights-off to sleep onset (rule chosen in Section 2; default = first non-W epoch).'),
    ('spt_min', 'Sleep period time (SPT)', 'min', 'macrostructure',
     'Sleep onset to the last sleep epoch, both included.'),
    ('tst_min', 'Total sleep time (TST)', 'min', 'macrostructure',
     'Number of N1 + N2 + N3 + R epochs within the sleep period x epoch length.'),
    ('waso_min', 'Wake after sleep onset (WASO)', 'min', 'macrostructure',
     'Number of W epochs within the sleep period x epoch length.'),
    ('other_min', 'Other epochs within SPT', 'min', 'macrostructure',
     'MT / custom / unknown epochs within the sleep period (neither sleep nor wake; see policy).'),
    ('se_pct', 'Sleep efficiency (SE)', '%', 'macrostructure', '100 x TST / TIB.'),
    ('sme_pct', 'Sleep maintenance efficiency (SME)', '%', 'macrostructure', '100 x TST / SPT.'),
    ('lat_n1_min', 'Latency to N1', 'min', 'macrostructure',
     'Sleep onset to the first N1 epoch (0 under the default onset rule when onset is N1).'),
    ('lat_n2_min', 'Latency to N2', 'min', 'macrostructure', 'Sleep onset to the first N2 epoch.'),
    ('lat_n3_min', 'Latency to N3', 'min', 'macrostructure', 'Sleep onset to the first N3 epoch.'),
    ('lat_rem_min', 'REM latency', 'min', 'macrostructure', 'Sleep onset to the first R epoch.'),
    ('n1_min', 'N1 duration', 'min', 'macrostructure', 'Number of N1 epochs within SPT x epoch length.'),
    ('n2_min', 'N2 duration', 'min', 'macrostructure', 'Number of N2 epochs within SPT x epoch length.'),
    ('n3_min', 'N3 duration', 'min', 'macrostructure', 'Number of N3 epochs within SPT x epoch length.'),
    ('rem_min', 'REM duration', 'min', 'macrostructure', 'Number of R epochs within SPT x epoch length.'),
    ('n1_pct', 'N1 (% of TST)', '%', 'macrostructure', '100 x N1 duration / TST.'),
    ('n2_pct', 'N2 (% of TST)', '%', 'macrostructure', '100 x N2 duration / TST.'),
    ('n3_pct', 'N3 (% of TST)', '%', 'macrostructure', '100 x N3 duration / TST.'),
    ('rem_pct', 'REM (% of TST)', '%', 'macrostructure', '100 x REM duration / TST.'),
    ('n_apnea_obstructive', 'Obstructive apneas', 'count', 'respiratory',
     'Scored obstructive apneas within the counting window.'),
    ('n_apnea_central', 'Central apneas', 'count', 'respiratory',
     'Scored central apneas within the counting window.'),
    ('n_apnea_mixed', 'Mixed apneas', 'count', 'respiratory',
     'Scored mixed apneas within the counting window.'),
    ('n_hypopnea', 'Hypopneas', 'count', 'respiratory',
     'Scored hypopneas (any canonical label starting with "hypopnea") within the counting window.'),
    ('oai', 'Obstructive apnea index (OAI)', 'events/h', 'respiratory', 'Obstructive apneas / TST (h).'),
    ('cai', 'Central apnea index (CAI)', 'events/h', 'respiratory', 'Central apneas / TST (h).'),
    ('mai', 'Mixed apnea index (MAI)', 'events/h', 'respiratory', 'Mixed apneas / TST (h).'),
    ('hi', 'Hypopnea index (HI)', 'events/h', 'respiratory', 'Hypopneas / TST (h).'),
    ('ahi', 'Apnea-hypopnea index (AHI)', 'events/h', 'respiratory',
     '(all apneas + hypopneas) / TST (h) = OAI + CAI + MAI + HI.'),
    ('n_plm', 'Periodic limb movements', 'count', 'plm',
     'Scored PLM events (canonical "plm") within the counting window.'),
    ('plm_index', 'PLM index', 'events/h', 'plm', 'PLM events / TST (h).'),
    ('n_arousal', 'Arousals', 'count', 'arousal',
     'Scored arousals (any canonical label starting with "arousal") within the counting window.'),
    ('arousal_index', 'Arousal index', 'events/h', 'arousal', 'Arousals / TST (h).'),
    ('n_desaturation', 'SpO2 desaturations', 'count', 'odi',
     'Scored desaturations within the counting window (depth >= threshold when the depth is known).'),
    ('odi', 'Oxygen desaturation index (ODI)', 'events/h', 'odi', 'Desaturations / TST (h).'),
    ('t90_min', 'T90', 'min', 'spo2', 'Time with SpO2 < threshold (default 90 %) between lights-off and lights-on.'),
    ('t90_pct_tst', 'T90 (% of TST)', '%', 'spo2', '100 x T90 / TST.'),
    ('t88_sleep_min', 'T88 during sleep', 'min', 'spo2',
     'Time with SpO2 <= 88 % during the sleep epochs (N1/N2/N3/R) between lights-off and lights-on '
     '(ICSD-3-TR sleep-related hypoxemia, criterion A: >= 5 min).'),
    ('spo2_mean_pct', 'Mean SpO2', '%', 'spo2', 'Mean of the valid SpO2 samples in bed.'),
    ('spo2_nadir_pct', 'SpO2 nadir', '%', 'spo2', 'Minimum of the valid SpO2 samples in bed.'),
    ('spo2_artifact_pct', 'SpO2 artefact', '%', 'spo2',
     'Percentage of SpO2 samples in bed discarded as artefact (<= 0 or below the floor).'),
]
METRIC_KEYS = [m[0] for m in METRICS]
METRIC_INFO = {m[0]: {'label': m[1], 'unit': m[2], 'group': m[3], 'definition': m[4]} for m in METRICS}

# Provenance written with every participant (text columns of the global table)
PROVENANCE_KEYS = [
    'lights_source', 'lights_off_clock', 'lights_on_clock', 'lights_off_epoch', 'lights_on_epoch',
    'sleep_onset_rule', 'sleep_onset_epoch', 'other_stage_policy', 'event_count_window',
    'epoch_length_s', 'n_epochs_hypno', 'n_epochs_in_bed', 'n_mt_custom', 'event_source',
    'n_events_total', 'n_unmapped_events', 'n_events_outside_window', 'has_spo2', 'odi_threshold_pct',
    'odi_rule', 'spo2_channel', 'light_channel', 'recording_start', 'edf_duration_min', 'n_warnings']

# COPIED FROM tool 5 as well (INDICATIVE ONLY: used here to shade the figures and by the
# "indicative range" buttons, never as a diagnostic cut-off). (low, high, note)
REFERENCE_RANGES = {
    'sol_min':        (0, 30,   '~15-20 min at every adult age, < 10 min longer at 80 than at 20 y (Ohayon 2004); '
                                '> 30 min suggests difficulty initiating sleep'),
    'tst_min':        (360, 480, '~7.5 h at 20-25 y, ~6.7 h at 50 y, ~6 h after 75 y in the laboratory '
                                 '(-10 min per decade, Ohayon 2004)'),
    'waso_min':       (0, 30,   'young adults; +10 min per decade from 30 y (~40 min at 55 y, ~55-75 min at '
                                '65-75 y in healthy subjects, Ohayon 2004) - > 30 min is not abnormal after 50 y'),
    'se_pct':         (85, 100, '~95 % at 20 y, ~90 % at 45, ~85 % at 65, ~80 % after 75 y (-3 % per decade '
                                'after 40 y, Ohayon 2004); lower on a first laboratory night'),
    'sme_pct':        (90, 100, 'sleep continuity once asleep (convention)'),
    'lat_rem_min':    (50, 110, '~90 min at 25 y, ~70 at 65, ~55 at 90 y (Ohayon 2004); <= 15 min = sleep-onset '
                                'REM period (ICSD-3-TR, narcolepsy); < 50 min sometimes associated with depression; '
                                'prolonged by REM-suppressing medication'),
    'n1_pct':         (2, 8,    '~6 % in young adults, ~7 % and up to ~10 % in older adults (Ohayon 2004); '
                                'high N1 = fragmented sleep'),
    'n2_pct':         (45, 58,  '~50 % at 25 y, ~55 % at 65, ~58 % at 90 y (Ohayon 2004)'),
    'n3_pct':         (13, 23,  'young adults (~18-20 %); ~12 % at 45 y, ~5-10 % after 65 y (-2 % per decade, '
                                'Ohayon 2004); depends on the amplitude criterion of the scorer'),
    'rem_pct':        (18, 25,  '~21-22 % in young adults, ~18-19 % after 70 y (Ohayon 2004)'),
    'ahi':            (0, 5,    'AASM severity: < 5 normal, 5-15 mild, 15-30 moderate, > 30 severe'),
    'plm_index':      (0, 15,   'PLMD threshold (ICSD-3-TR): > 15/h in adults (> 5/h in children)'),
    'arousal_index':  (0, 15,   'convention: ~10/h in young adults, higher with age; > 15-20/h = fragmented sleep'),
    'odi':            (0, 5,    'convention: same severity bands as the AHI; state the threshold (3 % or 4 %)'),
    't90_pct_tst':    (0, 10,   'research convention (CT90), not an ICSD-3-TR criterion'),
    't88_sleep_min':  (0, 5,    'ICSD-3-TR sleep-related hypoxemia (G47.36), criterion A: >= 5 min with SpO2 <= 88 % '
                                'during sleep (adults; <= 90 % in children); criterion B (not explained by '
                                'hypoventilation / OSA) needs a clinical evaluation'),
    'spo2_mean_pct':  (94, 100, 'convention'),
    'spo2_nadir_pct': (88, 100, 'convention - not an ICSD-3-TR criterion: an isolated nadir <= 88 % is not '
                                'hypoxemia (see T88)'),
}


def _find_unicorn_gif():
    """The GIF lives in tools/images/; probe from the repo root or from tools/ (Voila launch cwd)."""
    cwd = Path(os.getcwd())
    for folder in [cwd / 'tools' / 'images', cwd / 'images', cwd.parent / 'tools' / 'images']:
        if (folder / UNICORN_GIF_NAME).is_file():
            return folder / UNICORN_GIF_NAME
    return None


In [ ]:
# =============================================================================
# Core functions - load the table, evaluate the criteria, draw the distributions
# =============================================================================

def load_global_table(path):
    """Read tool 5's global table. Returns (df, metrics) where metrics = list of
    (key, label, unit, group): the registered tool-5 metrics present, in METRICS order, then any
    other numeric column (provenance columns excluded)."""
    df = pd.read_csv(path, sep='\t', dtype={'file_id': str})
    if 'file_id' not in df.columns:
        raise ValueError('no file_id column: is this a tool-5 global_sleep_metrics.tsv?')
    df['file_id'] = df['file_id'].astype(str)

    metrics = []
    for key in METRIC_KEYS:
        if key in df.columns:
            df[key] = pd.to_numeric(df[key], errors='coerce')
            info = METRIC_INFO[key]
            metrics.append((key, info['label'], info['unit'], info['group']))
    for col in df.columns:
        if col == 'file_id' or col in METRIC_INFO or col in PROVENANCE_KEYS:
            continue
        values = pd.to_numeric(df[col], errors='coerce')
        # a numeric column = at least one number and no text left once converted
        if values.notna().any() and values.notna().sum() == df[col].notna().sum():
            df[col] = values
            metrics.append((col, col, '', 'other'))
    return df, metrics


def parse_bound(text):
    """'' -> (None, True); '4.5' or '4,5' -> (4.5, True); anything else -> (None, False)."""
    text = (text or '').strip().replace(',', '.')
    if text == '':
        return None, True
    try:
        value = float(text)
    except ValueError:
        return None, False
    if not np.isfinite(value):
        return None, False
    return value, True


def evaluate_criteria(df, criteria, nan_excludes):
    """criteria = list of {'metric', 'min', 'max'} (None = no bound).
    Returns (fails, reasons): fails = DataFrame of bools (one column per criterion metric),
    reasons = list (one string per participant, '' when kept)."""
    fails = pd.DataFrame(index=df.index)
    reason_parts = [[] for _ in range(len(df))]
    for crit in criteria:
        key, lo, hi = crit['metric'], crit['min'], crit['max']
        values = df[key]
        fail = pd.Series(False, index=df.index)
        if lo is not None:
            fail |= values < lo            # NaN < lo is False: a missing value never fails here
        if hi is not None:
            fail |= values > hi
        if nan_excludes:
            fail |= values.isna()
        fails[key] = fail
        for pos, (is_fail, v) in enumerate(zip(fail.to_numpy(), values.to_numpy())):
            if not is_fail:
                continue
            if pd.isna(v):
                reason_parts[pos].append(f'{key} missing')
            elif lo is not None and v < lo:
                reason_parts[pos].append(f'{key} = {v:.4g} < {lo:g}')
            else:
                reason_parts[pos].append(f'{key} = {v:.4g} > {hi:g}')
    reasons = ['; '.join(parts) for parts in reason_parts]
    return fails, reasons


def build_selection_table(df, criteria, fails, reasons):
    """One row per participant: file_id, excluded, n_criteria_violated, exclude_reason,
    fail_<metric> per criterion, then the criterion metrics' values."""
    n_violated = fails.sum(axis=1).astype(int) if len(fails.columns) else pd.Series(0, index=df.index)
    out = pd.DataFrame({'file_id': df['file_id'],
                        'excluded': n_violated > 0,
                        'n_criteria_violated': n_violated,
                        'exclude_reason': reasons})
    for crit in criteria:
        out['fail_' + crit['metric']] = fails[crit['metric']].astype(bool)
    for crit in criteria:
        out[crit['metric']] = df[crit['metric']]
    return out.reset_index(drop=True)


def _metric_title(key, metrics_by_key):
    label, unit = metrics_by_key[key][1], metrics_by_key[key][2]
    return f'{label} ({unit})' if unit else label


def plot_distributions(df, keys, metrics_by_key, criteria_by_metric, excluded, n_excluded_by,
                       preview=None):
    """Small multiples, one panel per metric in `keys`: histogram of all participants, a jittered
    strip of the participants below it (red = excluded by ANY criterion), the thresholds as dashed
    lines with the excluded side shaded, and tool 5's indicative range shaded in green.
    n_excluded_by = {metric: participants excluded by that criterion} (from evaluate_criteria)."""
    n = len(keys)
    ncols = min(3, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.3 * nrows), squeeze=False)
    rng = np.random.default_rng(0)          # fixed seed: the dots do not jump between redraws
    jitter = rng.uniform(-1, 1, len(df))
    excluded = np.asarray(excluded, dtype=bool)

    for i, key in enumerate(keys):
        ax = axes[i // ncols][i % ncols]
        values = df[key].to_numpy(dtype=float)
        valid = np.isfinite(values)
        crit = criteria_by_metric.get(key)
        title = _metric_title(key, metrics_by_key)
        if key == preview and crit is None:
            title += '  [preview]'
        ax.set_title(title, fontsize=10, loc='left')
        for side in ('top', 'right'):
            ax.spines[side].set_visible(False)
        if not valid.any():
            ax.text(0.5, 0.5, 'no values', ha='center', va='center', transform=ax.transAxes,
                    color=COLOR_MUTED)
            ax.set_yticks([])
            continue

        # x range: data + thresholds + indicative range, so every line stays visible
        xs = list(values[valid])
        if crit is not None:
            xs += [b for b in (crit['min'], crit['max']) if b is not None]
        lo_x, hi_x = min(xs), max(xs)
        pad = (hi_x - lo_x) * 0.08 if hi_x > lo_x else max(abs(lo_x) * 0.1, 1.0)
        x0, x1 = lo_x - pad, hi_x + pad

        counts, _, _ = ax.hist(values[valid], bins=min(30, max(5, valid.sum() // 2)),
                               range=(lo_x, hi_x) if hi_x > lo_x else None,
                               color=COLOR_HIST, edgecolor='white', linewidth=1)
        top = max(counts.max(), 1)
        strip_y = -0.25 * top            # participants drawn under the histogram
        kept_v = valid & ~excluded
        excl_v = valid & excluded
        ax.scatter(values[kept_v], strip_y + jitter[kept_v] * 0.08 * top, s=22, color=COLOR_KEPT,
                   zorder=3, label='kept', edgecolors='white', linewidths=0.6)
        ax.scatter(values[excl_v], strip_y + jitter[excl_v] * 0.08 * top, s=26, color=COLOR_EXCLUDED,
                   zorder=4, label='excluded (any criterion)', edgecolors='white', linewidths=0.6)

        rng_ref = REFERENCE_RANGES.get(key)
        if rng_ref is not None:
            ax.axvspan(rng_ref[0], rng_ref[1], color=COLOR_BAND, alpha=0.10, zorder=0,
                       label='indicative range')
        if crit is not None:
            if crit['min'] is not None:
                ax.axvspan(x0, crit['min'], color=COLOR_EXCLUDED, alpha=0.08, zorder=0)
                ax.axvline(crit['min'], color=COLOR_EXCLUDED, linestyle='--', linewidth=1.6)
                ax.text(crit['min'], top * 1.02, f' < {crit["min"]:g}', color=COLOR_EXCLUDED,
                        fontsize=8, ha='left', va='bottom')
            if crit['max'] is not None:
                ax.axvspan(crit['max'], x1, color=COLOR_EXCLUDED, alpha=0.08, zorder=0)
                ax.axvline(crit['max'], color=COLOR_EXCLUDED, linestyle='--', linewidth=1.6)
                ax.text(crit['max'], top * 1.02, f'> {crit["max"]:g} ', color=COLOR_EXCLUDED,
                        fontsize=8, ha='right', va='bottom')
            ax.text(1.0, 1.0, f'{n_excluded_by.get(key, 0)} excluded by this criterion', transform=ax.transAxes,
                    ha='right', va='bottom', fontsize=8, color=COLOR_EXCLUDED)
        n_nan = int((~valid).sum())
        if n_nan:
            ax.text(0.0, -0.22, f'{n_nan} missing value(s)', transform=ax.transAxes, fontsize=8,
                    color=COLOR_MUTED)
        ax.set_xlim(x0, x1)
        ax.set_ylim(strip_y - 0.18 * top, top * 1.15)
        # integer participant counts, only over the histogram (not the strip below it)
        ax.set_yticks([t for t in MaxNLocator(integer=True, nbins=4).tick_values(0, top) if 0 <= t <= top])
        ax.set_ylabel('participants', fontsize=8, color=COLOR_MUTED)
        ax.tick_params(labelsize=8, colors=COLOR_MUTED)

    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')
    # one shared legend (identity is never carried by colour alone)
    handles, labels = [], []
    for ax in axes.flat:
        for h, l in zip(*ax.get_legend_handles_labels()):
            if l not in labels:
                handles.append(h)
                labels.append(l)
    if handles:
        fig.legend(handles, labels, loc='upper center', ncol=len(labels), frameon=False, fontsize=9,
                   bbox_to_anchor=(0.5, 1.0))
    fig.tight_layout(rect=(0, 0, 1, 0.95 if nrows == 1 else 0.97))
    return fig


def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')
    return base64.b64encode(buf.getvalue()).decode('ascii')


## 1. Load the macrostructure table

Pick the **data folder** (the one given to tool 5): the tool looks for
`reports_features_macrostructure/global_sleep_metrics.tsv` under it. To use another table (e.g. a
versioned `reports_features_macrostructure_v2/`), pick the `.tsv` directly with the second chooser,
which takes precedence. If a previous selection was validated for this table, its criteria are reloaded.

In [ ]:
# =============================================================================
# Section 1 - load the global table
# =============================================================================
S = {'df': None, 'metrics': [], 'table_path': None, 'building': False}
_LBL = {'description_width': 'initial'}

fc_data = FileChooser(os.getcwd())
fc_data.show_only_dirs = True
fc_data.title = ('<b>Data folder</b> (holds <code>reports_features_macrostructure/</code> '
                 'written by tool 5):')

fc_table = FileChooser(os.getcwd())
fc_table.filter_pattern = ['*.tsv']
fc_table.title = ('<b>Or pick the table directly</b> (optional, takes precedence): '
                  '<code>global_sleep_metrics.tsv</code>')

btn_load = widgets.Button(description='Load', button_style='primary', icon='upload')
lbl_load = widgets.HTML('<i>Pick the data folder (or the table), then Load.</i>')


def resolve_table_path():
    """The directly-picked table first, else <data>/reports_features_macrostructure/global_sleep_metrics.tsv."""
    if fc_table.selected:
        return Path(fc_table.selected)
    if fc_data.selected_path:
        return Path(fc_data.selected_path) / REPORTS_DIRNAME / GLOBAL_TABLE_NAME
    return None


def read_previous_criteria(table_path):
    """Criteria of a previous validation next to the table: ({metric: (min, max)}, nan_excludes)
    or (None, False). Non-fatal."""
    path = Path(table_path).parent / CRITERIA_TSV_NAME
    if not path.is_file():
        return None, False
    prev = pd.read_csv(path, sep='\t', dtype={'metric': str})
    prefill = {}
    for _, row in prev.iterrows():
        if row['metric'] == '(none)':
            continue
        lo = None if pd.isna(row['min']) else float(row['min'])
        hi = None if pd.isna(row['max']) else float(row['max'])
        prefill[row['metric']] = (lo, hi)
    nan_excl = bool(prev['nan_excludes'].iloc[0]) if len(prev) and 'nan_excludes' in prev else False
    return prefill, nan_excl


def on_load(_):
    try:
        path = resolve_table_path()
        if path is None:
            lbl_load.value = '<span style="color:#d03b3b">Pick a data folder or a table first.</span>'
            return
        if not path.is_file():
            lbl_load.value = (f'<span style="color:#d03b3b">Not found: <code>{html.escape(str(path))}</code>. '
                              'Run tool 5 first, or pick the table directly.</span>')
            return
        df, metrics = load_global_table(path)
        if not metrics:
            lbl_load.value = '<span style="color:#d03b3b">No numeric metric column found in this table.</span>'
            return
        S['df'], S['metrics'], S['table_path'] = df, metrics, path

        msg_prev = ''
        prefill, nan_excl = None, False
        try:
            prefill, nan_excl = read_previous_criteria(path)
        except Exception as e:
            msg_prev = (f'<br><span style="color:#e08a00">&#9888; could not read the previous criteria '
                        f'({html.escape(str(e))}): starting empty.</span>')
        if prefill:
            unknown = [m for m in prefill if m not in df.columns]
            msg_prev = (f'<br>&#8634; Previous selection found: <b>{len(prefill)}</b> criteria reloaded from '
                        f'<code>{CRITERIA_TSV_NAME}</code>.')
            if unknown:
                msg_prev += (f' <span style="color:#e08a00">&#9888; ignored (not in this table): '
                             f'{html.escape(", ".join(unknown))}</span>')

        n_nan = {k: int(df[k].isna().sum()) for k, *_ in metrics if df[k].isna().any()}
        nan_txt = ''
        if n_nan:
            nan_txt = ('<br><small>Missing values: ' +
                       ', '.join(f'{k} ({v})' for k, v in n_nan.items()) + '</small>')
        lbl_load.value = (f'&#10004; <b>{len(df)}</b> participants, <b>{len(metrics)}</b> metrics from '
                          f'<code>{html.escape(str(path))}</code>{msg_prev}{nan_txt}')
        build_criteria_ui(metrics, prefill or {}, nan_excl)
    except Exception as e:
        lbl_load.value = f'<span style="color:#d03b3b">&#10060; Load failed: {html.escape(str(e))}</span>'


btn_load.on_click(on_load)
display(widgets.VBox([fc_data, fc_table, widgets.HBox([btn_load]), lbl_load]))


## 2. Criteria and live distributions

Tick a metric and type **`exclude if <`** and/or **`exclude if >`** (leave a field blank for no bound;
typing a value ticks the row). **Indicative range** fills both fields with tool 5's healthy-adult range
(indicative only: age-, medication- and first-night-dependent, adapt it to your study). An invalid
value turns the field red and is ignored.

The figure redraws at every change: one panel per enabled criterion (plus the metric chosen in
**Preview**, to look at a distribution before deciding). Grey bars = histogram of all participants, dots
below = one per participant (**red = excluded by any criterion**, so you see where the participants
removed by one criterion sit on the other metrics), dashed red lines = your thresholds, green band =
indicative range.

In [ ]:
# =============================================================================
# Section 2 - criteria widgets, live evaluation and plot
# =============================================================================
ROWS = {}                 # metric -> {'cb', 'tmin', 'tmax', 'btn', 'lbl', 'group'}
ACC_GROUPS = []           # group names in accordion order

acc_criteria = widgets.Accordion(children=[])
cb_nan = widgets.Checkbox(value=False, description='Missing value excludes (metric not computed = violation)',
                          indent=False, layout=widgets.Layout(width='520px'))
dd_preview = widgets.Dropdown(description='Preview:', options=[('(none)', None)], value=None,
                              layout=widgets.Layout(width='420px'), style=_LBL)
btn_clear = widgets.Button(description='Clear all criteria', icon='eraser')
lbl_summary = widgets.HTML('')
out_plot = widgets.Output()
html_excluded = widgets.HTML('')
_BORDER_BAD = '2px solid #d03b3b'


def build_criteria_ui(metrics, prefill, nan_excludes):
    """(Re)build one row per metric, grouped by metric group, pre-filled from a previous selection."""
    S['building'] = True
    try:
        ROWS.clear()
        ACC_GROUPS.clear()
        by_group = {}
        for key, label, unit, group in metrics:
            by_group.setdefault(group if group in GROUP_ORDER else 'other', []).append((key, label, unit))
        children = []
        for group in GROUP_ORDER:
            if group not in by_group:
                continue
            rows = []
            for key, label, unit in by_group[group]:
                lo, hi = prefill.get(key, (None, None))
                cb = widgets.Checkbox(value=key in prefill, indent=False, layout=widgets.Layout(width='28px'))
                name = widgets.HTML(f'<b>{html.escape(label)}</b>'
                                    + (f' <small>({html.escape(unit)})</small>' if unit else '')
                                    + f'<br><small style="color:#898781">{html.escape(key)}</small>',
                                    layout=widgets.Layout(width='280px'))
                tmin = widgets.Text(value='' if lo is None else f'{lo:g}', description='exclude if <',
                                    layout=widgets.Layout(width='190px'), style=_LBL)
                tmax = widgets.Text(value='' if hi is None else f'{hi:g}', description='exclude if >',
                                    layout=widgets.Layout(width='190px'), style=_LBL)
                ref = REFERENCE_RANGES.get(key)
                btn = widgets.Button(description='indicative range', disabled=ref is None,
                                     tooltip=(f'{ref[0]:g}-{ref[1]:g}: {ref[2]}' if ref else
                                              'no indicative range for this metric'),
                                     layout=widgets.Layout(width='130px'))
                lbl = widgets.HTML('', layout=widgets.Layout(width='190px'))
                ROWS[key] = {'cb': cb, 'tmin': tmin, 'tmax': tmax, 'btn': btn, 'lbl': lbl, 'group': group}
                cb.observe(refresh, names='value')
                tmin.observe(lambda ch, k=key: _on_bound_typed(k, ch), names='value')
                tmax.observe(lambda ch, k=key: _on_bound_typed(k, ch), names='value')
                btn.on_click(lambda _, k=key: _on_indicative(k))
                rows.append(widgets.HBox([cb, name, tmin, tmax, btn, lbl],
                                         layout=widgets.Layout(align_items='center')))
            children.append(widgets.VBox(rows))
            ACC_GROUPS.append(group)
        acc_criteria.children = children
        for i, group in enumerate(ACC_GROUPS):
            acc_criteria.set_title(i, group)
        cb_nan.value = bool(nan_excludes)
        dd_preview.options = [('(none)', None)] + [(f'{label} [{key}]', key) for key, label, _, _ in metrics]
        dd_preview.value = None
    finally:
        S['building'] = False
    refresh()


def _on_bound_typed(key, change):
    # typing a valid bound in an unticked row ticks it (the tick triggers the refresh)
    value, ok = parse_bound(change['new'])
    if ok and value is not None and not ROWS[key]['cb'].value and not S['building']:
        ROWS[key]['cb'].value = True
    else:
        refresh()


def _on_indicative(key):
    try:
        ref = REFERENCE_RANGES.get(key)
        if ref is None:
            return
        row = ROWS[key]
        S['building'] = True
        try:
            # a range starting at 0 (e.g. AHI 0-5) has no meaningful lower bound
            row['tmin'].value = '' if ref[0] == 0 else f'{ref[0]:g}'
            # an upper limit of 100 % (SE, SpO2) is not a bound either
            row['tmax'].value = '' if (ref[1] == 100 and key.endswith('_pct')) else f'{ref[1]:g}'
            row['cb'].value = True
        finally:
            S['building'] = False
        refresh()
    except Exception as e:
        lbl_summary.value = f'<span style="color:#d03b3b">&#10060; {html.escape(str(e))}</span>'


def current_criteria():
    """Ticked rows with at least one valid bound -> list of {'metric', 'min', 'max'}.
    Also colours invalid fields red."""
    criteria = []
    for key, row in ROWS.items():
        lo, ok_lo = parse_bound(row['tmin'].value)
        hi, ok_hi = parse_bound(row['tmax'].value)
        row['tmin'].layout.border = '' if ok_lo else _BORDER_BAD
        row['tmax'].layout.border = '' if ok_hi else _BORDER_BAD
        if row['cb'].value and (lo is not None or hi is not None):
            criteria.append({'metric': key, 'min': lo, 'max': hi})
    return criteria


def compute_selection():
    """Evaluate the current criteria on the loaded table -> (criteria, fails, reasons, excluded mask)."""
    df = S['df']
    criteria = current_criteria()
    fails, reasons = evaluate_criteria(df, criteria, cb_nan.value)
    excluded = fails.any(axis=1).to_numpy() if len(fails.columns) else np.zeros(len(df), dtype=bool)
    return criteria, fails, reasons, excluded


def refresh(*_):
    if S['building'] or S['df'] is None:
        return
    try:
        df = S['df']
        criteria, fails, reasons, excluded = compute_selection()
        crit_by_metric = {c['metric']: c for c in criteria}

        # per-row counts and accordion titles
        n_active = {g: 0 for g in ACC_GROUPS}
        for key, row in ROWS.items():
            if key in crit_by_metric:
                n = int(fails[key].sum())
                row['lbl'].value = f'<span style="color:#d03b3b">&#8594; {n} excluded</span>'
                n_active[row['group']] += 1
            elif row['cb'].value:
                row['lbl'].value = '<small style="color:#898781">no bound yet</small>'
            else:
                row['lbl'].value = ''
        for i, group in enumerate(ACC_GROUPS):
            acc_criteria.set_title(i, f'{group}  ({n_active[group]} active)' if n_active[group] else group)

        n_excl = int(excluded.sum())
        lbl_summary.value = (f'<div style="font-size:1.1em;margin:6px 0"><b>{len(df) - n_excl}</b> kept / '
                             f'{len(df)} participants &nbsp;(<b style="color:#d03b3b">{n_excl}</b> excluded, '
                             f'{len(criteria)} active criteria)</div>')

        if n_excl:
            rows_html = ''.join(
                f'<tr><td style="padding:2px 10px"><b>{html.escape(fid)}</b></td>'
                f'<td style="padding:2px 10px">{html.escape(reason)}</td></tr>'
                for fid, reason, ex in zip(df['file_id'], reasons, excluded) if ex)
            html_excluded.value = ('<b>Excluded participants</b><table style="border-collapse:collapse">'
                                   f'{rows_html}</table>')
        else:
            html_excluded.value = ''

        keys = [c['metric'] for c in criteria]
        if dd_preview.value is not None and dd_preview.value not in keys:
            keys.append(dd_preview.value)
        with out_plot:
            clear_output(wait=True)
            if keys:
                metrics_by_key = {m[0]: m for m in S['metrics']}
                n_by = {k: int(fails[k].sum()) for k in fails.columns}
                fig = plot_distributions(df, keys, metrics_by_key, crit_by_metric, excluded, n_by,
                                         preview=dd_preview.value)
                display(fig)
                plt.close(fig)
            else:
                display(HTML('<i>Enable a criterion or choose a metric in Preview to see its distribution.</i>'))
    except Exception as e:
        lbl_summary.value = f'<span style="color:#d03b3b">&#10060; Refresh failed: {html.escape(str(e))}</span>'


def on_clear(_):
    try:
        S['building'] = True
        try:
            for row in ROWS.values():
                row['tmin'].value = ''
                row['tmax'].value = ''
                row['cb'].value = False
        finally:
            S['building'] = False
        refresh()
    except Exception as e:
        lbl_summary.value = f'<span style="color:#d03b3b">&#10060; {html.escape(str(e))}</span>'


cb_nan.observe(refresh, names='value')
dd_preview.observe(refresh, names='value')
btn_clear.on_click(on_clear)

display(widgets.VBox([acc_criteria, widgets.HBox([cb_nan, dd_preview, btn_clear]),
                      lbl_summary, out_plot, html_excluded]))


## 3. Validate the selection

Add an optional comment (why these criteria: protocol, inclusion criteria of the study...), then
**Validate**. The three output files are written beside the loaded table and **replace** a previous
selection for that table (its criteria were reloaded in Section 1, so nothing is lost by accident).

In [ ]:
# =============================================================================
# Section 3 - validate: write the selection, the criteria and the report
# =============================================================================
txt_comment = widgets.Textarea(description='Comment:', placeholder='optional: why these criteria',
                               layout=widgets.Layout(width='700px', height='60px'), style=_LBL)
btn_validate = widgets.Button(description='Validate selection', button_style='success', icon='check')
lbl_validate = widgets.HTML('')
out_unicorn = widgets.Output()


def build_report_html(df, sel, crit_table, criteria, excluded, table_path, validated_at, comment):
    metrics_by_key = {m[0]: m for m in S['metrics']}
    n_excl = int(sel['excluded'].sum())
    parts = ['<html><head><meta charset="utf-8"><title>Participant selection</title><style>'
             'body{font-family:Segoe UI,Arial,sans-serif;margin:24px;color:#0b0b0b;background:#fcfcfb}'
             'table{border-collapse:collapse;font-size:13px}td,th{border:1px solid #ddd;padding:3px 8px}'
             'th{background:#f1f0ec}.excl{color:#d03b3b;font-weight:bold}</style></head><body>',
             '<h1>Participant selection on sleep macrostructure criteria</h1>',
             f'<p>Table: <code>{html.escape(str(table_path))}</code><br>Validated: {validated_at}<br>'
             f'<b>{len(sel) - n_excl}</b> kept / {len(sel)} participants '
             f'(<span class="excl">{n_excl} excluded</span>), {len(criteria)} criteria. '
             f'Missing value excludes: {cb_nan.value}.</p>']
    if comment:
        parts.append(f'<p><b>Comment:</b> {html.escape(comment)}</p>')
    parts.append('<p><i>Rule: a participant is excluded when any criterion is violated '
                 '(value &lt; min or value &gt; max). The indicative ranges (green) come from tool 5 and '
                 'are not diagnostic cut-offs.</i></p>')
    parts.append('<h2>Criteria</h2>' + crit_table.to_html(index=False, na_rep=''))
    if criteria:
        fig = plot_distributions(df, [c['metric'] for c in criteria], metrics_by_key,
                                 {c['metric']: c for c in criteria}, excluded,
                                 dict(zip(crit_table['metric'], crit_table['n_excluded_by'])))
        parts.append('<h2>Distributions</h2>'
                     f'<img style="max-width:100%" src="data:image/png;base64,{fig_to_base64(fig)}">')
        plt.close(fig)
    excl_rows = sel[sel['excluded']][['file_id', 'exclude_reason']]
    parts.append('<h2>Excluded participants</h2>'
                 + (excl_rows.to_html(index=False) if len(excl_rows) else '<p>None.</p>'))
    parts.append('<h2>All participants</h2>' + sel.to_html(index=False, na_rep=''))
    parts.append('</body></html>')
    return '\n'.join(parts)


def show_unicorn(n_kept, n_total):
    with out_unicorn:
        clear_output()
        if not SHOW_UNICORN:
            return
        caption = widgets.HTML(f'<div style="text-align:center;font-size:1.4em;color:#7a4fc9">'
                               f'<b>Selection validated &#10024;</b><br>'
                               f'<span style="font-size:0.8em">{n_kept} / {n_total} participants kept</span></div>')
        gif = _find_unicorn_gif()
        if gif is not None:
            img = widgets.Image(value=gif.read_bytes(), format='gif', width=260)
        else:                       # GIF missing: the emoji does the job
            img = widgets.HTML('<div style="font-size:120px;text-align:center">&#129412;</div>')
        display(widgets.VBox([img, caption], layout=widgets.Layout(align_items='center', width='340px')))


def on_validate(_):
    out_unicorn.clear_output()
    try:
        if S['df'] is None:
            lbl_validate.value = '<span style="color:#d03b3b">Load a table first (Section 1).</span>'
            return
        df = S['df']
        criteria, fails, reasons, excluded = compute_selection()
        validated_at = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        comment = txt_comment.value.strip()
        out_dir = Path(S['table_path']).parent
        # Windows refuses paths of 260 characters or more: say so instead of a cryptic "file not found"
        longest = max(len(str(out_dir / name)) for name in (SELECTION_TSV_NAME, CRITERIA_TSV_NAME, REPORT_HTML_NAME))
        if longest >= 260:
            lbl_validate.value = (f'<span style="color:#d03b3b">&#10060; Output path too long for Windows '
                                  f'({longest} characters, limit 259): move or rename the data folder to a '
                                  f'shorter path. Nothing was written.</span>')
            return

        sel = build_selection_table(df, criteria, fails, reasons)
        metrics_by_key = {m[0]: m for m in S['metrics']}
        crit_rows = [{'metric': c['metric'], 'label': metrics_by_key[c['metric']][1],
                      'unit': metrics_by_key[c['metric']][2], 'min': c['min'], 'max': c['max'],
                      'n_excluded_by': int(fails[c['metric']].sum())} for c in criteria]
        if not crit_rows:           # no criterion: everyone kept, still record the decision
            crit_rows = [{'metric': '(none)', 'label': 'no criterion', 'unit': '', 'min': None,
                          'max': None, 'n_excluded_by': 0}]
        crit_table = pd.DataFrame(crit_rows)
        crit_table['nan_excludes'] = cb_nan.value
        crit_table['source_table'] = str(S['table_path'])
        crit_table['validated_at'] = validated_at
        crit_table['comment'] = comment

        # data first (the durable record), then the report
        sel.to_csv(out_dir / SELECTION_TSV_NAME, sep='\t', index=False, na_rep='n/a')
        crit_table.to_csv(out_dir / CRITERIA_TSV_NAME, sep='\t', index=False, na_rep='')
        msg = (f'&#10004; Written in <code>{html.escape(str(out_dir))}</code>: <code>{SELECTION_TSV_NAME}</code>, '
               f'<code>{CRITERIA_TSV_NAME}</code>')
        try:
            report = build_report_html(df, sel, crit_table, criteria, excluded, S['table_path'],
                                       validated_at, comment)
            (out_dir / REPORT_HTML_NAME).write_text(report, encoding='utf-8')
            msg += f', <code>{REPORT_HTML_NAME}</code>.'
        except Exception as e:
            msg += (f'.<br><span style="color:#e08a00">&#9888; report not written: '
                    f'{html.escape(str(e))}</span>')
        n_excl = int(excluded.sum())
        lbl_validate.value = msg + f'<br><b>{len(df) - n_excl}</b> kept, <b>{n_excl}</b> excluded.'
        show_unicorn(len(df) - n_excl, len(df))
    except Exception as e:
        lbl_validate.value = f'<span style="color:#d03b3b">&#10060; Validation failed: {html.escape(str(e))}</span>'


btn_validate.on_click(on_validate)
display(widgets.VBox([txt_comment, btn_validate, lbl_validate, out_unicorn]))


## Outputs: where to look

All three files sit beside the loaded `global_sleep_metrics.tsv` (in `reports_features_macrostructure/`).

- **`participant_selection.tsv`**: the decision, one row per participant. Read `excluded` (True/False,
  joined on `file_id`) and `exclude_reason` (e.g. `ahi = 27.3 > 15; tst_min = 212 < 240`). The
  `fail_<metric>` columns say which criterion each participant violated, followed by the values used.
- **`participant_selection_criteria.tsv`**: the criteria (`metric`, `min`, `max`, `n_excluded_by`), the
  missing-value policy, the source table, the date and your comment. Reloaded by Section 1 next time.
- **`participant_selection_report.html`**: the same, readable, with the distribution figure: the file
  to show a colleague or to archive with the study.

Before trusting a decision, check in the report that each threshold falls where you expected on the
distribution, and that a participant excluded for a *missing* value (if that option was ticked) really
lacks the measure (e.g. no SpO2 channel) rather than having failed in tool 5 (see its
`sleep_metrics_failed.tsv`).